# 09 — Search Results: Coverage and Language Distribution

Analysis of the GitHub search results produced by running the multilingual search pipeline against the reviewed search terms from `reviewed_grouped_translated_terms.csv`.

Three result types are available: **repositories**, **issues**, and **users**. Each was searched by term string; results carry `search_term`, `natural_language` (the comma-separated language codes that share that term), and standard GitHub API fields.

This notebook asks:

- How many results does each search type return, and how many unique items survive deduplication?
- What fraction of the 4,285 kept terms returned any results at all?
- How many of the 880 pipeline languages have at least one result of any type?
- **What does the multilingual approach add over English-only or a handful of major languages?**
- Which language families drive the most results, and which are only reachable through non-English terms?

**Sections**
1. Data loading and deduplication
2. Search term hit rate
3. The value of multilingual search
4. Language and family coverage
5. Result distribution by language family

In [1]:
import os
import sys
import glob

import pandas as pd
import altair as alt
from pathlib import Path

alt.data_transformers.enable("vegafusion")

sys.path.insert(0, str(Path("..").resolve()))
from scripts.utils import get_data_directory_path, get_language_family

DATA_DIR     = get_data_directory_path()
TERM         = "Digital Humanities"
TERM_SLUG    = TERM.lower().replace(" ", "_")

# Search results live outside the pipeline datasets directory
SEARCH_DIR   = Path(DATA_DIR).parent.parent / "new_datasets"
REPO_GLOB    = str(SEARCH_DIR / "searched_repo_data"  / "digital_humanities" / "*.csv")
ISSUE_GLOB   = str(SEARCH_DIR / "searched_issue_data" / "digital_humanities" / "*.csv")
USER_GLOB    = str(SEARCH_DIR / "searched_user_data"  / "digital_humanities" / "*.csv")

SEARCH_TERMS_PATH = os.path.join(
    DATA_DIR, "translated_terms", TERM_SLUG,
    "search_terms", "reviewed_grouped_translated_terms.csv"
)
DISAGR_PATH  = os.path.join(
    DATA_DIR, "translated_terms", TERM_SLUG, "evaluation", "disagreement_analysis.csv"
)

print(f"Search dir: {SEARCH_DIR}")
print(f"Search terms: {SEARCH_TERMS_PATH}")

Retrieving translation pipeline data directory path...

Search dir: /Users/zleblanc/CodingDH/new_datasets
Search terms: /Users/zleblanc/CodingDH/translation_transmogrification_pipeline/datasets/translated_terms/digital_humanities/search_terms/reviewed_grouped_translated_terms.csv


In [2]:
# ── Manual exclusions (search notebook: both types apply) ───────────────────
from scripts.utils import load_manual_exclusions

_excl_eval_dir = os.path.join(DATA_DIR, "translated_terms", TERM_SLUG, "evaluation")
analysis_langs, search_terms, corrections = load_manual_exclusions(_excl_eval_dir)
print(f"Manual exclusions loaded:")
print(f"  analysis_exclusion : {len(analysis_langs)} language codes  (dropped entirely)")
print(f"  search_exclusion   : {len(search_terms)} (language, term) pairs  (also excluded from search)")
print(f"  term_correction    : {len(corrections)} corrections")
print()
print("Note: search results for both analysis-excluded languages and search-excluded terms")
print("      are excluded from this notebook's analysis.")

Manual exclusions loaded:
  analysis_exclusion : 84 language codes  (dropped entirely)
  search_exclusion   : 209 (language, term) pairs  (also excluded from search)
  term_correction    : 82 corrections

Note: search results for both analysis-excluded languages and search-excluded terms
      are excluded from this notebook's analysis.


## 8.1 — Data Loading and Deduplication

Each result type is spread across one CSV per search term. Files are concatenated and deduplicated by GitHub item `id`. The pre-dedup row count reflects how many terms found each item (a repo discovered by three different search terms appears three times before dedup).

In [3]:
def load_results(file_glob: str, id_col: str = "id") -> pd.DataFrame:
    files = glob.glob(file_glob)
    if not files:
        return pd.DataFrame()
    return pd.concat(
        [pd.read_csv(f, low_memory=False) for f in files],
        ignore_index=True,
    )

repos_raw  = load_results(REPO_GLOB)
issues_raw = load_results(ISSUE_GLOB)
users_raw  = load_results(USER_GLOB)

# Filter to current reviewed terms only — excludes stale result files
_kept_terms = set(
    pd.read_csv(SEARCH_TERMS_PATH)
    .query("keep_term == True")["search_term"]
    .str.lower().str.strip()
)
repos_raw  = repos_raw[repos_raw["search_term"].str.lower().str.strip().isin(_kept_terms)]
issues_raw = issues_raw[issues_raw["search_term"].str.lower().str.strip().isin(_kept_terms)]
users_raw  = users_raw[users_raw["search_term"].str.lower().str.strip().isin(_kept_terms)]

repos  = repos_raw.drop_duplicates("id").copy()
issues = issues_raw.drop_duplicates("id").copy()
users  = users_raw.drop_duplicates("id").copy()

summary = pd.DataFrame([
    {"type": "Repositories", "raw_rows": len(repos_raw),  "unique_items": len(repos),
     "terms_that_hit": repos_raw["search_term"].nunique(),
     "avg_term_hits": round(len(repos_raw) / len(repos), 1)},
    {"type": "Issues",       "raw_rows": len(issues_raw), "unique_items": len(issues),
     "terms_that_hit": issues_raw["search_term"].nunique(),
     "avg_term_hits": round(len(issues_raw) / len(issues), 1)},
    {"type": "Users",        "raw_rows": len(users_raw),  "unique_items": len(users),
     "terms_that_hit": users_raw["search_term"].nunique(),
     "avg_term_hits": round(len(users_raw) / len(users), 1)},
])
print(summary.to_string(index=False))

        type  raw_rows  unique_items  terms_that_hit  avg_term_hits
Repositories     33270          4722              94            7.0
      Issues     23160          6351             127            3.6
       Users      5156          1543              33            3.3


In [21]:
repos.search_term.value_counts().head(20)

search_term
Digital Humanities        1425
Humanities Digital        1229
digital humanitie          562
digital humanities         397
buku digital               161
informatica umanistica     146
artes digitales             95
Humanités numériques        87
Humanidades Digital         82
digi-humani                 69
新浪财经                        64
Humanités Numériques        55
humani digital              45
Digihumanitaaria            38
Humanidades Digitais        32
not available               29
humanî digital              28
tidak tersedia              27
数字人文                        21
sena digital                18
Name: count, dtype: int64

In [22]:
users.search_term.value_counts().head(20)

search_term
Digital Humanities        1029
digital humanities         388
Informatica umanistica      30
Humanidades Digital         26
Humanidades Digitais        20
artes digitales             15
Humanidades digitales       11
informatica umanistica       5
Humanités Numeriques         3
Humanidades digitais         3
tidak tersedia               3
dijital sanatlar             2
buku digital                 1
Digital humaniora            1
Sastra Digital               1
Digitaalhumanitaaria         1
Humanities Digital           1
humanidades digitales        1
Humanidades Digitales        1
hadothi                      1
Name: count, dtype: int64

In [23]:
issues.search_term.value_counts().head(20)

search_term
tidak tersedia                    1810
Humanities Digital                1377
digital a humanities               644
数字人文                               472
Digital Humanities                 439
nhân văn số                        205
电子文学                               204
数化人文                               172
ŋɔɔŋ ŋɔɔŋ ŋɔɔŋ ŋɔɔŋ                126
Nhân văn số                        120
人工智能文学                             106
humanidades diġital                 69
Humanités Numeriques                69
buku digital                        56
humanities digital                  35
Humanidades Digitais                30
數字文學                                28
i cannot provide a translation      26
数码人文                                25
Humaniora Digital                   24
Name: count, dtype: int64

In [7]:
# Load search terms and build helper maps
st = pd.read_csv(SEARCH_TERMS_PATH)
st_kept = st[st["keep_term"] == True].copy()
st_kept["search_term_lower"] = st_kept["search_term"].str.lower().str.strip()

disagr = pd.read_csv(DISAGR_PATH, converters={"language_code": str})

# Explode natural_language (comma-separated) → individual language codes
# Filters out multi-word artifacts like 'see also: test languages ...'
lang_term_rows = [] 
for _, row in st_kept.iterrows():
    codes = [
        c.strip() for c in str(row["natural_language"]).split(",")
        if c.strip() and " " not in c.strip()
    ]
    for code in codes:
        lang_term_rows.append({"language_code": code,
                               "search_term_lower": row["search_term_lower"],
                               "category": row["category"]})
lang_term_map = pd.DataFrame(lang_term_rows)

# Attach language family
lang_meta = disagr[["language_code", "language_name", "language_family"]].drop_duplicates("language_code")
lang_term_map = lang_term_map.merge(lang_meta, on="language_code", how="left")

print(f"Kept search terms: {len(st_kept)}")
print(f"Unique language codes mapped: {lang_term_map['language_code'].nunique()}")

Kept search terms: 4285
Unique language codes mapped: 795


## 8.2 — Search Term Hit Rate

Of the 4,285 kept terms, the vast majority return no GitHub results — which is expected. GitHub indexes public content in the language it was written; only a small fraction of the world's DH activity is on GitHub, and even less uses a term that matches a specific translation. The hit rate tells us which terms are productive and whether there is a long tail.

In [8]:
# Count results per search term across all types
def term_hit_counts(df: pd.DataFrame, label: str) -> pd.DataFrame:
    counts = (
        df.groupby(df["search_term"].str.lower().str.strip())["id"]
        .nunique()
        .reset_index()
    )
    counts.columns = ["search_term_lower", f"n_{label}"]
    return counts

repo_hits  = term_hit_counts(repos_raw,  "repos")
issue_hits = term_hit_counts(issues_raw, "issues")
user_hits  = term_hit_counts(users_raw,  "users")

hit_df = (
    st_kept[["search_term_lower", "category"]]
    .merge(repo_hits,  on="search_term_lower", how="left")
    .merge(issue_hits, on="search_term_lower", how="left")
    .merge(user_hits,  on="search_term_lower", how="left")
    .fillna(0)
)
for col in ["n_repos", "n_issues", "n_users"]:
    hit_df[col] = hit_df[col].astype(int)
hit_df["any_hit"] = (hit_df[["n_repos", "n_issues", "n_users"]] > 0).any(axis=1)
hit_df["total"]   = hit_df["n_repos"] + hit_df["n_issues"] + hit_df["n_users"]

print(f"Terms with ≥1 result of any type : {hit_df['any_hit'].sum()} / {len(hit_df)} ({hit_df['any_hit'].mean()*100:.1f}%)")
print(f"Terms with ≥1 repo               : {(hit_df['n_repos']>0).sum()}")
print(f"Terms with ≥1 issue              : {(hit_df['n_issues']>0).sum()}")
print(f"Terms with ≥1 user               : {(hit_df['n_users']>0).sum()}")
print()
print("Top 15 terms by total results:")
print(hit_df.nlargest(15, "total")[["search_term_lower","n_repos","n_issues","n_users","total"]].to_string(index=False))

Terms with ≥1 result of any type : 125 / 4285 (2.9%)
Terms with ≥1 repo               : 63
Terms with ≥1 issue              : 101
Terms with ≥1 user               : 18

Top 15 terms by total results:
    search_term_lower  n_repos  n_issues  n_users  total
   digital humanities     3672      2490     1417   7579
   humanities digital     1333      2487       17   3837
          digi-humani     2032         0        0   2032
    digital humanitie     1954         0        0   1954
       tidak tersedia       27      1810        3   1840
       humani digital     1379         7        0   1386
     humaniti digital     1325         0        0   1325
       humanî digital     1315         6        0   1321
 digital a humanities        1       661        0    662
                 数字人文       27       528        0    555
          nhân văn số        0       330        0    330
humanidades digitales      139        69       40    248
 humanités numériques      145        83        3    231
  

In [9]:
# Distribution of result counts per term (log-scale for the long tail)
hist_data = pd.DataFrame({
    "type": ["Repos"] * len(hit_df) + ["Issues"] * len(hit_df) + ["Users"] * len(hit_df),
    "count": list(hit_df["n_repos"]) + list(hit_df["n_issues"]) + list(hit_df["n_users"]),
})
hist_data = hist_data[hist_data["count"] > 0]

base = alt.Chart(hist_data).mark_bar(opacity=0.7).encode(
    x=alt.X("count:Q", bin=alt.Bin(maxbins=40), title="results per term"),
    y=alt.Y("count():Q", title="number of terms",
            scale=alt.Scale(type="log", domain=[1, None])),
    color=alt.Color("type:N", scale=alt.Scale(scheme="tableau10")),
).properties(width=200, height=160)

hist = base.facet(facet="type:N", columns=3,
                  title="Results per search term (log y-axis)")
display(hist)

alt.FacetChart(...)

### Top Productive Terms by Script Direction and Category

Which specific search terms drive the most results, and does script direction (LTR vs RTL) predict productivity? RTL scripts — Arabic, Hebrew, Persian, Urdu — represent communities that English-only search misses entirely. The disagreement `category` assigned during translation review may also predict term quality: COMPLETE_CONSENSUS terms, where all services agreed, should be more reliable than TRANSMOGRIFICATION or STRUCTURAL_ABSENCE terms.

In [11]:
# Merge directionality into hit_df
hit_dir = hit_df.merge(
    st_kept[["search_term_lower", "directionality"]].drop_duplicates("search_term_lower"),
    on="search_term_lower", how="left",
)
hit_dir["dir_label"] = hit_dir["directionality"].apply(
    lambda d: "RTL" if str(d).strip().lower() == "rtl" else "LTR"
)

# ── Top 25 terms: stacked repos / issues / users ──────────────────────────
top25 = hit_dir.nlargest(25, "total").copy()
top25_melt = top25.melt(
    id_vars=["search_term_lower", "dir_label", "total"],
    value_vars=["n_repos", "n_issues", "n_users"],
    var_name="result_type", value_name="n",
)
top25_melt["result_type"] = top25_melt["result_type"].str.replace("n_", "").str.capitalize()
type_order = ["Repos", "Issues", "Users"]

top_bar = alt.Chart(top25_melt).mark_bar().encode(
    y=alt.Y("search_term_lower:N",
            sort=alt.EncodingSortField(field="total", order="descending"),
            title=None),
    x=alt.X("n:Q", title="results"),
    color=alt.Color("result_type:N", sort=type_order,
                    scale=alt.Scale(domain=type_order, scheme="tableau10"),
                    title="Result type"),
    order=alt.Order("result_type:N", sort="ascending"),
    tooltip=[
        alt.Tooltip("search_term_lower:N", title="term"),
        alt.Tooltip("dir_label:N", title="direction"),
        alt.Tooltip("result_type:N", title="type"),
        alt.Tooltip("n:Q", title="count"),
        alt.Tooltip("total:Q", title="total"),
    ],
).properties(width=430, height=480, title="Top 25 terms by total results")

# Direction stripe on the left axis
dir_tick = alt.Chart(top25).mark_tick(thickness=4, size=18).encode(
    y=alt.Y("search_term_lower:N",
            sort=alt.EncodingSortField(field="total", order="descending"),
            axis=None),
    x=alt.value(0),
    color=alt.Color("dir_label:N",
                    scale=alt.Scale(domain=["LTR", "RTL"],
                                    range=["#1f77b4", "#d62728"]),
                    title="Direction"),
    tooltip=["search_term_lower:N", "dir_label:N"],
)

display((dir_tick | top_bar).resolve_scale(y="shared"))

# Summary counts
n_ltr = (top25["dir_label"] == "LTR").sum()
n_rtl = (top25["dir_label"] == "RTL").sum()
print(f"Top 25 breakdown: {n_ltr} LTR, {n_rtl} RTL")

alt.HConcatChart(...)

Top 25 breakdown: 25 LTR, 0 RTL


In [13]:
# ── LTR vs RTL comparison ─────────────────────────────────────────────────
dir_summary = (
    hit_dir.groupby("dir_label")
    .agg(
        total_terms=("search_term_lower", "count"),
        terms_with_hits=("any_hit", "sum"),
        total_results=("total", "sum"),
    )
    .reset_index()
)
dir_summary["hit_rate_pct"] = (
    dir_summary["terms_with_hits"] / dir_summary["total_terms"] * 100
).round(1)
dir_summary["avg_per_hit"] = (
    dir_summary["total_results"] / dir_summary["terms_with_hits"].replace(0, pd.NA)
).round(1)

print("LTR vs RTL summary:")
print(dir_summary[["dir_label","total_terms","terms_with_hits","hit_rate_pct",
                    "total_results","avg_per_hit"]].to_string(index=False))

dir_metrics = [
    ("hit_rate_pct",  "Hit rate (%)"),
    ("total_results", "Total results"),
    ("avg_per_hit",   "Avg results / hit term"),
]
dir_charts = []
for col, title in dir_metrics:
    c = alt.Chart(dir_summary).mark_bar().encode(
        x=alt.X("dir_label:N", title=None),
        y=alt.Y(f"{col}:Q", title=title),
        color=alt.Color("dir_label:N",
                        scale=alt.Scale(domain=["LTR", "RTL"],
                                        range=["#1f77b4", "#d62728"]),
                        legend=None),
        tooltip=["dir_label:N", alt.Tooltip(f"{col}:Q", title=title)],
    ).properties(width=130, height=180, title=title)
    dir_charts.append(c)
display(alt.hconcat(*dir_charts))

# ── Top 10 RTL terms ──────────────────────────────────────────────────────
rtl_top = hit_dir[hit_dir["dir_label"] == "RTL"].nlargest(10, "total")
if len(rtl_top):
    print("\nTop 10 RTL terms:")
    print(rtl_top[["search_term_lower","n_repos","n_issues","n_users","total"]].to_string(index=False))
else:
    print("\nNo RTL terms with results.")

# ── Category breakdown ────────────────────────────────────────────────────
# Each grouped term may span multiple categories; take primary (alphabetically first)
hit_dir["primary_cat"] = hit_dir["category"].str.split(",").str[0].str.strip()

cat_summary = (
    hit_dir.groupby("primary_cat")
    .agg(
        total_terms=("search_term_lower", "count"),
        terms_with_hits=("any_hit", "sum"),
        total_results=("total", "sum"),
    )
    .reset_index()
)
cat_summary["hit_rate_pct"] = (
    cat_summary["terms_with_hits"] / cat_summary["total_terms"] * 100
).round(1)
cat_summary = cat_summary.sort_values("total_results", ascending=False)

cat_results = alt.Chart(cat_summary).mark_bar().encode(
    y=alt.Y("primary_cat:N", sort="-x", title=None),
    x=alt.X("total_results:Q", title="total results"),
    color=alt.Color("hit_rate_pct:Q", scale=alt.Scale(scheme="blues"),
                    title="hit rate (%)"),
    tooltip=["primary_cat:N", "total_terms:Q", "terms_with_hits:Q",
             "hit_rate_pct:Q", "total_results:Q"],
).properties(width=300, height=160, title="Total results by category")

cat_rate = alt.Chart(cat_summary).mark_bar().encode(
    y=alt.Y("primary_cat:N", sort="-x", title=None, axis=alt.Axis(labels=False)),
    x=alt.X("hit_rate_pct:Q", title="hit rate (%)"),
    color=alt.Color("total_terms:Q", scale=alt.Scale(scheme="greys"),
                    title="total terms"),
    tooltip=["primary_cat:N", "total_terms:Q", "hit_rate_pct:Q"],
).properties(width=200, height=160, title="Hit rate by category")

display(cat_results | cat_rate)
print()
print(cat_summary.to_string(index=False))

LTR vs RTL summary:
dir_label  total_terms  terms_with_hits  hit_rate_pct  total_results  avg_per_hit
      LTR         3995              124           3.1          26532        214.0
      RTL          290                1           0.3             67         67.0


alt.HConcatChart(...)


Top 10 RTL terms:
       search_term_lower  n_repos  n_issues  n_users  total
                    新浪财经       64         3        0     67
         ڈیجیٹل انسانیات        0         0        0      0
العلوم الإنسانية الرقمية        0         0        0      0
        ڈیجیٹل ہیومینٹیز        0         0        0      0
       الإنسانية الدقيقة        0         0        0      0
     علوم انسانی دیجیتال        0         0        0      0
      الإنسانيات الرقمية        0         0        0      0
  العلوم الرقمية للحضارة        0         0        0      0
          ڊجيٽل انسانيات        0         0        0      0
       دیجیتال انسانیاتی        0         0        0      0


alt.HConcatChart(...)


            primary_cat  total_terms  terms_with_hits  total_results  hit_rate_pct
PRODUCTIVE_DISAGREEMENT          150               25          12145          16.7
     STRUCTURAL_ABSENCE         2118               71          10982           3.4
     TRANSMOGRIFICATION         2010               23           2874           1.1
     COMPLETE_CONSENSUS            4                4            596         100.0
   MEASUREMENT_ARTEFACT            3                2              2          66.7


## 8.3 — The Value of Multilingual Search

The core methodological question: how much does searching in 800+ languages add over searching in English only, or a manually curated shortlist of major languages?

Three baselines are compared:
- **English only** — search term `"digital humanities"` alone
- **Major 11 languages** — English plus the 10 most-spoken languages in the pipeline (Spanish, French, German, Portuguese, simplified Chinese, Japanese, Korean, Arabic, Russian, Hindi), represented by their ISO 639-1 codes
- **Full multilingual** — all 4,285 kept terms

Because a single GitHub item can be found by multiple search terms, the analysis checks which term(s) found each item across **all** raw rows before deduplication.

In [14]:
MAJOR_LANG_CODES = {"en", "es", "fr", "de", "pt", "zh", "ja", "ko", "ar", "ru", "hi"}

# Build set of search terms that include at least one major language code
major_terms = set()
en_terms    = set()
for _, row in st_kept.iterrows():
    codes = {c.strip() for c in str(row["natural_language"]).split(",") if c.strip()}
    if "en" in codes:
        en_terms.add(row["search_term_lower"])
    if codes & MAJOR_LANG_CODES:
        major_terms.add(row["search_term_lower"])

def value_add(raw_df: pd.DataFrame, id_col: str = "id") -> dict:
    """For each result type, count unique ids reached by each baseline."""
    norm = raw_df["search_term"].str.lower().str.strip()
    en_ids    = set(raw_df[norm.isin(en_terms)][id_col].dropna())
    major_ids = set(raw_df[norm.isin(major_terms)][id_col].dropna())
    all_ids   = set(raw_df[id_col].dropna())
    return {
        "english_only":          len(en_ids),
        "major_11_languages":    len(major_ids),
        "full_multilingual":     len(all_ids),
        "added_over_english":    len(all_ids - en_ids),
        "added_over_major":      len(all_ids - major_ids),
        "english_pct":           round(len(en_ids)    / len(all_ids) * 100, 1) if all_ids else 0,
        "major_pct":             round(len(major_ids) / len(all_ids) * 100, 1) if all_ids else 0,
    }

value_rows = []
for label, raw_df in [("Repositories", repos_raw), ("Issues", issues_raw), ("Users", users_raw)]:
    row = value_add(raw_df)
    row["type"] = label
    value_rows.append(row)

value_df = pd.DataFrame(value_rows).set_index("type")
print(value_df[["english_only","major_11_languages","full_multilingual",
                "added_over_english","added_over_major",
                "english_pct","major_pct"]].to_string())

              english_only  major_11_languages  full_multilingual  added_over_english  added_over_major  english_pct  major_pct
type                                                                                                                           
Repositories          3805                3981               4722                 917               741         80.6       84.3
Issues                2555                2653               6351                3796              3698         40.2       41.8
Users                 1420                1482               1543                 123                61         92.0       96.0


In [15]:
# Stacked bar: English-only / major-only / multilingual-only split per result type
stacked_rows = []
for label, raw_df in [("Repositories", repos_raw), ("Issues", issues_raw), ("Users", users_raw)]:
    norm      = raw_df["search_term"].str.lower().str.strip()
    en_ids    = set(raw_df[norm.isin(en_terms)]["id"].dropna())
    major_ids = set(raw_df[norm.isin(major_terms)]["id"].dropna())
    all_ids   = set(raw_df["id"].dropna())
    stacked_rows += [
        {"type": label, "segment": "English only",            "n": len(en_ids)},
        {"type": label, "segment": "Major langs, not English", "n": len(major_ids - en_ids)},
        {"type": label, "segment": "Full multilingual only",   "n": len(all_ids - major_ids)},
    ]

seg_order = ["English only", "Major langs, not English", "Full multilingual only"]
stacked_df = pd.DataFrame(stacked_rows)

bar = alt.Chart(stacked_df).mark_bar().encode(
    y=alt.Y("type:N", title=None),
    x=alt.X("n:Q", title="unique items"),
    color=alt.Color("segment:N", sort=seg_order,
                    scale=alt.Scale(range=["#1f77b4", "#ff7f0e", "#2ca02c"]),
                    title="Found by"),
    order=alt.Order("segment:N", sort="ascending"),
    tooltip=["type:N", "segment:N", "n:Q"],
).properties(width=450, height=140,
             title="Unique results by search scope — English only vs. major languages vs. full multilingual")
display(bar)

# Pct labels
for _, row in stacked_df.iterrows():
    total = stacked_df[stacked_df["type"] == row["type"]]["n"].sum()
    print(f"  {row['type']:15s} {row['segment']:35s}: {row['n']:5d} ({row['n']/total*100:.1f}%)")

alt.Chart(...)

  Repositories    English only                       :  3805 (80.6%)
  Repositories    Major langs, not English           :   176 (3.7%)
  Repositories    Full multilingual only             :   741 (15.7%)
  Issues          English only                       :  2555 (40.2%)
  Issues          Major langs, not English           :    98 (1.5%)
  Issues          Full multilingual only             :  3698 (58.2%)
  Users           English only                       :  1420 (92.0%)
  Users           Major langs, not English           :    62 (4.0%)
  Users           Full multilingual only             :    61 (4.0%)


## 8.4 — Language and Family Coverage

How many of the 880 pipeline languages have at least one result? And which languages contribute uniquely — i.e., results that would not have been found without that language's translation?

In [16]:
# For each result type: map raw rows → individual language codes via lang_term_map
def attach_languages(raw_df: pd.DataFrame) -> pd.DataFrame:
    """Join raw results to individual language codes via the search-term map."""
    norm = raw_df.copy()
    norm["search_term_lower"] = norm["search_term"].str.lower().str.strip()
    return norm.merge(
        lang_term_map[["search_term_lower", "language_code", "language_name", "language_family"]],
        on="search_term_lower", how="left",
    )

repos_lang  = attach_languages(repos_raw)
issues_lang = attach_languages(issues_raw)
users_lang  = attach_languages(users_raw)

# Languages with ≥1 result per type
for label, df in [("Repos", repos_lang), ("Issues", issues_lang), ("Users", users_lang)]:
    n = df["language_code"].dropna().nunique()
    print(f"{label}: {n}/880 languages with ≥1 result ({n/880*100:.1f}%)")

# Any type
all_lang_codes = (
    pd.concat([repos_lang, issues_lang, users_lang])["language_code"]
    .dropna().unique()
)
print(f"Any type: {len(all_lang_codes)}/880 languages with ≥1 result ({len(all_lang_codes)/880*100:.1f}%)")

Repos: 172/880 languages with ≥1 result (19.5%)
Issues: 206/880 languages with ≥1 result (23.4%)
Users: 77/880 languages with ≥1 result (8.8%)
Any type: 245/880 languages with ≥1 result (27.8%)


In [17]:
# Per-language repo count — top 30
lang_repo_counts = (
    repos_lang.groupby(["language_code", "language_name", "language_family"])["id"]
    .nunique()
    .reset_index(name="n_repos")
    .sort_values("n_repos", ascending=False)
)
print("Top 30 languages by unique repo count:")
print(lang_repo_counts.head(30).to_string(index=False))

Top 30 languages by unique repo count:
language_code              language_name                 language_family  n_repos
           en                    English         Indo-European languages     3805
          xmn  Manichaean Middle Persian         Indo-European languages     2032
          gaa                         Ga     Niger-Kordofanian languages     1954
           mh                Marshallese          Austronesian languages     1379
           id                 Indonesian          Austronesian languages     1340
          gor                  Gorontalo          Austronesian languages     1334
          nia                       Nias          Austronesian languages     1334
          mwv                   Mentawai          Austronesian languages     1333
          dtp              Central Dusun          Austronesian languages     1328
          iba                       Iban          Austronesian languages     1326
           ms                      Malay          Austrones

In [18]:
# Languages that contribute results NOT found by English
en_repo_ids = set(repos_raw[repos_raw["search_term"].str.lower().str.strip().isin(en_terms)]["id"].dropna())

unique_by_lang = (
    repos_lang[~repos_lang["id"].isin(en_repo_ids)]
    .groupby(["language_code", "language_name", "language_family"])["id"]
    .nunique()
    .reset_index(name="n_unique_repos")
    .sort_values("n_unique_repos", ascending=False)
)
print(f"Languages contributing repos NOT found by English: {len(unique_by_lang)}")
print()
print("Top 20:")
print(unique_by_lang.head(20).to_string(index=False))

Languages contributing repos NOT found by English: 162

Top 20:
language_code            language_name                   language_family  n_unique_repos
           qu                  Quechua   South American Indian languages             167
          ext             Extremaduran           Indo-European languages             167
          lad   Ladino / Judeo-Spanish           Indo-European languages             167
          mwl                Mirandese           Indo-European languages             167
          kea             Kabuverdianu               Creoles and pidgins             164
          tet                    Tetum            Austronesian languages             164
          lbw                   Tolaki            Austronesian languages             162
           it                  Italian           Indo-European languages             146
           an                Aragonese           Indo-European languages             132
          ast                 Asturian        

## 8.5 — Result Distribution by Language Family

Aggregating results to the language family level shows which parts of the world's DH activity are captured by the multilingual approach, and which families only appear because of the full translation coverage (i.e., they would be missed by English-only or major-language search).

In [19]:
# Unique repos per language family
fam_repos = (
    repos_lang.groupby("language_family")["id"]
    .nunique()
    .reset_index(name="n_repos")
    .dropna(subset=["language_family"])
    .sort_values("n_repos", ascending=False)
)

# Flag which families are reachable via English / major languages
en_families    = set(repos_lang[repos_lang["id"].isin(en_repo_ids)]["language_family"].dropna())
major_repo_ids = set(repos_raw[repos_raw["search_term"].str.lower().str.strip().isin(major_terms)]["id"].dropna())
major_families = set(repos_lang[repos_lang["id"].isin(major_repo_ids)]["language_family"].dropna())

fam_repos["reachable_by_english"] = fam_repos["language_family"].isin(en_families)
fam_repos["reachable_by_major"]   = fam_repos["language_family"].isin(major_families)
fam_repos["multilingual_only"]    = ~fam_repos["reachable_by_major"]

# Compute color category in pandas — nested alt.condition not supported in Altair v6
def _reach_label(row):
    if row["multilingual_only"]:
        return "Multilingual only"
    if row["reachable_by_english"]:
        return "English"
    return "Major langs"

fam_repos["reach"] = fam_repos.apply(_reach_label, axis=1)

fam_bar = alt.Chart(fam_repos).mark_bar().encode(
    y=alt.Y("language_family:N", sort="-x", title=None),
    x=alt.X("n_repos:Q", title="unique repositories"),
    color=alt.Color("reach:N",
                    scale=alt.Scale(
                        domain=["English", "Major langs", "Multilingual only"],
                        range=["#1f77b4", "#ff7f0e", "#2ca02c"],
                    ),
                    title="Reachable by"),
    tooltip=["language_family:N", "n_repos:Q", "reach:N"],
).properties(width=400, height=380,
             title="Unique repos per language family — blue=English, orange=major langs, green=multilingual only")
display(fam_bar)

print(f"Families reachable by English only        : {len(en_families)}")
print(f"Families reachable by major 11 languages  : {len(major_families)}")
print(f"Families only reachable via full pipeline : {fam_repos['multilingual_only'].sum()}")
if fam_repos["multilingual_only"].any():
    print("\nMultilingual-only families:")
    for _, r in fam_repos[fam_repos["multilingual_only"]].iterrows():
        print(f"  {r['language_family']}: {r['n_repos']} repos")

alt.Chart(...)

Families reachable by English only        : 14
Families reachable by major 11 languages  : 18
Families only reachable via full pipeline : 3

Multilingual-only families:
  Afro-Asiatic languages: 64 repos
  Khoisan languages: 1 repos
  Sign languages: 1 repos


In [20]:
# Cross-type summary: repos + issues + users per family
fam_issues = (
    issues_lang.groupby("language_family")["id"]
    .nunique().reset_index(name="n_issues").dropna(subset=["language_family"])
)
fam_users = (
    users_lang.groupby("language_family")["id"]
    .nunique().reset_index(name="n_users").dropna(subset=["language_family"])
)

fam_all = (
    fam_repos[["language_family","n_repos"]]
    .merge(fam_issues, on="language_family", how="outer")
    .merge(fam_users,  on="language_family", how="outer")
    .fillna(0)
)
for col in ["n_repos","n_issues","n_users"]:
    fam_all[col] = fam_all[col].astype(int)
fam_all["total"] = fam_all["n_repos"] + fam_all["n_issues"] + fam_all["n_users"]
fam_all = fam_all.sort_values("total", ascending=False)

print("Results by language family (all types):")
print(fam_all.to_string(index=False))

Results by language family (all types):
                  language_family  n_repos  n_issues  n_users  total
          Indo-European languages     4320      2738     1534   8592
           Austronesian languages     1881      5093       90   7064
      Niger-Kordofanian languages     2300      1030       69   3399
           Sino-Tibetan languages       39       965        0   1004
             Artificial languages       73       620       24    717
              Creoles and pidgins      340       202       68    610
              Tai-Kadai languages       27       553        0    580
             Hmong-Mien languages       27       530        0    557
  South American Indian languages      187       127       64    378
         Austro-Asiatic languages        2       347        0    349
  North American Indian languages      168        70       40    278
           Nilo-Saharan languages      134       132        0    266
Central American Indian languages      139        69       40  